In [ ]:
import os
import shutil
import matplotlib.pyplot as plt
import cv2
import numpy as np
import random
import shutil
import json
import pandas as pd
import random
from skimage.color import label2rgb
from matplotlib.path import Path as pat
from pathlib import Path
from tqdm import tqdm
from utils.utils import read_config, get_image_in_folder
from utils.Image_processing import split_image_into_patch

config = read_config("./config.yml")

In [ ]:
path_to_image = Path("./datasets/prepared_dataset/")
paths_to_images = get_image_in_folder(path_to_image)

In [ ]:
# images_path = []
images_path = [ip for ip in paths_to_images if "image" in ip.as_posix()]
mask_paths = [ip for ip in paths_to_images if "masks" in ip.as_posix()]

In [ ]:
mask_paths

In [ ]:
def count_contours_per_class(
    mask_paths: list[str], 
    min_area: int = 10, 
    verbose: bool = True
) -> dict[int, int]:

    contour_counts = {config.task.labels[class_id]: 0 for class_id in range(len(config.task.labels))}
    mask_iterator = tqdm(mask_paths, desc="Counting contours") if verbose else mask_paths

    for mask_path in mask_iterator:
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        for class_id in range(len(config.task.labels)):
            binary_mask = (mask == class_id).astype(np.uint8) * 255

            contours, _ = cv2.findContours(
                binary_mask, 
                cv2.RETR_EXTERNAL,  
                cv2.CHAIN_APPROX_SIMPLE
            )

            valid_contours = [
                cnt for cnt in contours 
                if cv2.contourArea(cnt) >= min_area
            ]

            contour_counts[config.task.labels[class_id]] += len(valid_contours)

    return contour_counts

In [ ]:
contour_counts = count_contours_per_class(mask_paths)

In [ ]:
def count_pixel_per_class(
    mask_paths: list[str], 
    verbose: bool = True
) -> dict[int, int]:

    pixels_counts = {config.task.labels[class_id]: 0 for class_id in range(len(config.task.labels))}
    mask_iterator = tqdm(mask_paths, desc="Counting pixels") if verbose else mask_paths

    for mask_path in mask_iterator:
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        for class_id in range(len(config.task.labels)):
            binary_mask = (mask == class_id).astype(np.uint8) * 255
            
            pixel_count = cv2.countNonZero(binary_mask)
            pixels_counts[config.task.labels[class_id]] += pixel_count

    return pixels_counts

In [ ]:
pixels_counts = count_pixel_per_class(mask_paths)

In [ ]:
labels = list(pixel_counts.keys())
values = list(pixel_counts.values())

colors = plt.cm.Set3(np.linspace(0, 1, len(labels)))

plt.figure(figsize=(12, 6))
bars = plt.bar(labels, values, color=colors)
plt.title('Количество пикселей по классам')
plt.ylabel('Количество пикселей')
plt.xlabel('Классы')
plt.xticks(rotation=45)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:,.0f}'.replace(',', ' '),
             ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
labels = list(contour_counts.keys())
values = list(contour_counts.values())

colors = plt.cm.Set3(np.linspace(0, 1, len(labels)))

plt.figure(figsize=(12, 6))
bars = plt.bar(labels, values, color=colors)
plt.title('Количество контуров по классам')
plt.ylabel('Количество контуров')
plt.xlabel('Классы')
plt.xticks(rotation=45)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:,.0f}'.replace(',', ' '),
             ha='center', va='bottom')

plt.tight_layout()
plt.show()